# Model Training, Explainable AI (SHAP), and C++ Engine Verification

**Project Title**: Explainable Machine Learning Framework for Network Intrusion Detection and Feature-Based Threat Analysis  
**Objective**: Demonstrate leakage-free preprocessing, multi-model training, Tree SHAP attributions, and C++17 inference parity.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add project root to sys.path
project_root = Path.cwd().parent
sys.path.append(str(project_root / "src"))

from intrusion_xai.data.load_data import load_raw_data
from intrusion_xai.data.preprocess import prepare_data, NetworkDataPreprocessor
from intrusion_xai.ml.train import train_all_models
from intrusion_xai.ml.evaluate import evaluate_all_models
from intrusion_xai.explainability.global_explanations import compute_permutation_importance, compute_tree_shap_global
from intrusion_xai.explainability.local_explanations import explain_local_prediction
from intrusion_xai.integration.cpp_bridge import CppTreeEngineBridge
from intrusion_xai.config import MODELS_DIR, CPP_BINARY_PATH

print("All modules imported successfully.")

## 1. Load Data and Apply Leakage-Free Preprocessing

In [ ]:
data_splits = prepare_data(test_size=0.20, random_state=42)
X_train = data_splits["X_train"]
X_test = data_splits["X_test"]
y_train = data_splits["y_train"]
y_test = data_splits["y_test"]
feature_names = data_splits["feature_names"]

print(f"Training set shape: {X_train.shape} (Leakage-free scaled)")
print(f"Test set shape: {X_test.shape}")
print(f"Features retained after zero-variance drop: {len(feature_names)}")

## 2. Multi-Model Training and Serialization

In [ ]:
models = train_all_models(X_train, y_train, feature_names)
for name, model in models.items():
    print(f"Model trained and saved: {name}")

## 3. Comparative Evaluation & Confusion Metrics

In [ ]:
eval_results = evaluate_all_models(models, X_test, y_test)
metrics_df = pd.DataFrame(eval_results).T
metrics_df[["accuracy", "precision", "recall", "f1_score", "false_positive_rate", "false_negative_rate"]]

## 4. Explainable AI: Global Tree SHAP Feature Attribution

In [ ]:
dt_model = models["decision_tree"]
shap_results = compute_tree_shap_global(dt_model, X_test, feature_names)
top_feats = pd.DataFrame({
    "Feature": shap_results["top_features"],
    "Mean |SHAP|": shap_results["top_importance_values"]
})
print("Top 5 Discriminant Features Governing Network Intrusion:")
print(top_feats.head(5))

## 5. Local Explanation with Cybersecurity Domain Lexicon

In [ ]:
raw_df = load_raw_data()
preprocessor = data_splits["preprocessor"]

# Select a raw PortScan flow
sample_flow = raw_df[raw_df["Label"] != "BENIGN"].iloc[0]
local_explanations = explain_local_prediction(dt_model, preprocessor, sample_flow)

print("=== SOC Analyst Local Waterfall Attribution ===")
for item in local_explanations:
    print(f"[{item['threat_impact']}] {item['display_name']} (SHAP: {item['shap_value']:+.4f})")
    print(f"   Significance: {item['explanation']}")

## 6. Native C++17 Inference Engine & Parity Verification

In [ ]:
bridge = CppTreeEngineBridge()
print(f"C++ Engine Standard: {bridge.get_version()}")

# Compare first 100 test samples
matches = 0
sample_count = 100
py_preds = dt_model.predict(X_test[:sample_count])

for i in range(sample_count):
    cpp_pred = bridge.predict_single(X_test[i])
    if cpp_pred == py_preds[i]:
        matches += 1

print(f"Python vs. C++17 Parity: {matches}/{sample_count} ({matches/sample_count * 100:.2f}% Match)")